# Week 07 - Tuesday Assignment
## NLP Foundations: Embeddings, Sentiment, and Word2Vec
### Dataset: ShopSense E-Commerce Reviews

**Tasks:**
- Q1: Train Word2Vec on ShopSense reviews; explore polysemy of 'cheap'; build disambiguation system; compare window sizes
- Q2: Compute cosine similarity between Review A and Review B using BOW, TF-IDF, Word2Vec averaging, and Sentence-BERT

## Imports and Configuration

In [1]:
%pip install gensim

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [3]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

from gensim.models import Word2Vec
from gensim.utils import simple_preprocess

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer

DATA_PATH          = Path("../data/shopsense_reviews.csv")
TARGET_LANGUAGE    = "English"
W2V_VECTOR_SIZE    = 100
W2V_MIN_COUNT      = 2
W2V_EPOCHS         = 10
WINDOW_SMALL       = 2
WINDOW_LARGE       = 10
RANDOM_SEED        = 42
SBERT_MODEL_NAME   = "all-MiniLM-L6-v2"

# Anchor word pairs for 'cheap' disambiguation
ANCHOR_AFFORDABLE  = ["affordable", "budget", "value", "inexpensive", "economical"]
ANCHOR_LOW_QUALITY = ["flimsy", "poor", "inferior", "shoddy", "substandard"]

# The two fixed reviews for Q2
REVIEW_A = "incredible camera but terrible battery life"
REVIEW_B = "Battery drains fast, although photos are stunning"

print("Imports complete. Configuration loaded.")

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports complete. Configuration loaded.


## 1. Data Loading and Preprocessing

In [4]:
def load_reviews(path, language_filter=None):
    """
    Load the ShopSense reviews CSV and optionally filter by language.
    """
    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        raise FileNotFoundError(f"CSV not found at: {path}")

    required_columns = {"review_text", "language", "sentiment_label"}
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"CSV is missing required columns: {missing}")

    df = df.dropna(subset=["review_text"])

    if language_filter:
        df = df[df["language"] == language_filter].copy()

    print(f"Loaded {len(df):,} reviews"
          + (f" (language='{language_filter}')" if language_filter else ""))
    return df


In [5]:
def clean_text(text):
    """
    Remove HTML tags and non-ASCII punctuation from a single review string.
    """
    text = re.sub(r"<[^>]+>", " ", str(text))   # strip HTML tags
    text = re.sub(r"[^a-zA-Z0-9\s.,!?']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()


def tokenize_corpus(texts):
    """
    Apply gensim simple_preprocess to a list of raw text strings.
    """
    return [simple_preprocess(clean_text(t), deacc=True) for t in texts]


# Run the pipeline
reviews_df   = load_reviews(DATA_PATH, language_filter=TARGET_LANGUAGE)
tokenised    = tokenize_corpus(reviews_df["review_text"].tolist())

print(f"\nSample tokenised review: {tokenised[0][:10]}")

Loaded 7,219 reviews (language='English')

Sample tokenised review: ['do', 'not', 'buy', 'this', 'fake', 'product', 'nothing', 'like', 'the', 'images']


---
## Q1 -- Word2Vec on ShopSense Reviews
### Q1(a) -- Polysemy of 'cheap': one vector, two cosines

In [6]:
def train_word2vec(sentences, vector_size, window, min_count, epochs, seed):
    """
    Train a Word2Vec (skip-gram) model on the given tokenised sentences.
    """
    model = Word2Vec(
        sentences=sentences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        sg=1,          # skip-gram
        epochs=epochs,
        seed=seed,
        workers=1,     # deterministic with seed
    )
    vocab_size = len(model.wv)
    print(f"Word2Vec trained | window={window} | vocab size={vocab_size:,}")
    return model


def get_cosine_pair(model, word_a, word_b):
    """
    Return the cosine similarity between two words in the Word2Vec vocabulary.
    Returns None if either word is out-of-vocabulary.
    """
    vocab = model.wv.key_to_index
    if word_a not in vocab:
        print(f"  WARNING: '{word_a}' not in vocabulary.")
        return None
    if word_b not in vocab:
        print(f"  WARNING: '{word_b}' not in vocabulary.")
        return None
    return float(model.wv.similarity(word_a, word_b))


# Train the primary model (window=WINDOW_SMALL to start)
model_w2    = train_word2vec(
    tokenised,
    vector_size=W2V_VECTOR_SIZE,
    window=WINDOW_SMALL,
    min_count=W2V_MIN_COUNT,
    epochs=W2V_EPOCHS,
    seed=RANDOM_SEED,
)


Word2Vec trained | window=2 | vocab size=249


In [7]:

print()
print("Q1(a): Word2Vec gives ONE vector per word.")

# Show that 'cheap' is a single vector
PIVOT = "cheap"
if PIVOT in model_w2.wv.key_to_index:
    vec = model_w2.wv[PIVOT]
    print(f"\n'cheap' vector shape : {vec.shape}")
    print(f"'cheap' vector (first 10 dims): {np.round(vec[:10], 4)}")
    print()
    print("This single vector must carry BOTH senses (affordable AND low-quality).")
    print("It will be positioned somewhere between those two semantic poles.")
else:
    print(f"'{PIVOT}' is not in the vocabulary. Check that the CSV contains reviews with this word.")

# Cosine similarities
cos_affordable  = get_cosine_pair(model_w2, PIVOT, "affordable")
cos_flimsy      = get_cosine_pair(model_w2, PIVOT, "flimsy")

# Fallback: use nearest neighbours to find polysemy evidence
print("\nNearest neighbours of 'cheap' (top 10):")
if PIVOT in model_w2.wv.key_to_index:
    neighbours = model_w2.wv.most_similar(PIVOT, topn=10)
    for rank, (word, score) in enumerate(neighbours, 1):
        print(f"  {rank:2d}. {word:<20s}  sim={score:.4f}")

print()
if cos_affordable is not None:
    print(f"cosine('cheap', 'affordable') = {cos_affordable:.4f}")
else:
    print("'affordable' is OOV -- cosine cannot be computed.")

if cos_flimsy is not None:
    print(f"cosine('cheap', 'flimsy')     = {cos_flimsy:.4f}")
else:
    print("'flimsy' is OOV -- cosine cannot be computed.")



Q1(a): Word2Vec gives ONE vector per word.

'cheap' vector shape : (100,)
'cheap' vector (first 10 dims): [ 0.6466 -0.1612  0.0556  0.0818  0.2388 -0.1559  0.2709  0.5397 -0.5173
  0.3933]

This single vector must carry BOTH senses (affordable AND low-quality).
It will be positioned somewhere between those two semantic poles.

Nearest neighbours of 'cheap' (top 10):
   1. material              sim=0.9330
   2. finishing             sim=0.7781
   3. price                 sim=0.7638
   4. worth                 sim=0.7077
   5. thoda                 sim=0.6716
   6. poor                  sim=0.6253
   7. zyada                 sim=0.6193
   8. control               sim=0.6142
   9. job                   sim=0.6086
  10. much                  sim=0.5656

'affordable' is OOV -- cosine cannot be computed.
'flimsy' is OOV -- cosine cannot be computed.



### Interpretation:
Word2Vec produces a SINGLE, context-averaged vector for 'cheap'.
The corpus skews toward negative reviews, so 'cheap' leans toward
low-quality neighbours even though it can also mean affordable.
This is the core limitation of static embeddings for polysemous words.

### Q1(b) -- Disambiguation system: 'affordable' vs 'low-quality'

In [8]:
def build_anchor_vector(model, anchor_words):
    """
    Average the Word2Vec vectors of anchor words that exist in the vocabulary.
    """
    vectors = []
    for word in anchor_words:
        if word in model.wv.key_to_index:
            vectors.append(model.wv[word])
        else:
            print(f"  Anchor word '{word}' not in vocab -- skipping.")

    if not vectors:
        raise ValueError("No anchor words found in vocabulary.")

    return np.mean(vectors, axis=0)


def get_context_vector(model, sentence, pivot_word):
    """
    Compute the average Word2Vec vector of all context words in a sentence,
    excluding the pivot word itself.
    """
    tokens = simple_preprocess(clean_text(sentence), deacc=True)
    context_tokens = [t for t in tokens if t != pivot_word]

    vectors = []
    for token in context_tokens:
        if token in model.wv.key_to_index:
            vectors.append(model.wv[token])

    if not vectors:
        return None

    return np.mean(vectors, axis=0)

In [9]:


def cosine_between_vectors(vec_a, vec_b):
    """
    Compute cosine similarity between two numpy vectors.
    """
    vec_a = vec_a.reshape(1, -1)
    vec_b = vec_b.reshape(1, -1)
    return float(cosine_similarity(vec_a, vec_b)[0, 0])


def disambiguate_cheap(model, sentence, pivot_word,
                       anchor_affordable_words, anchor_low_quality_words):
    """
    Determine whether 'pivot_word' is used in the 'affordable' or 'low-quality'
    sense in the given sentence by comparing context vectors to anchor centroids.
    """
    result = {
        "sentence": sentence,
        "context_vector": None,
        "sim_affordable": None,
        "sim_low_quality": None,
        "predicted_sense": "unknown",
    }

    context_vec = get_context_vector(model, sentence, pivot_word)
    if context_vec is None:
        print(f"  No in-vocabulary context words found for: '{sentence}'")
        return result

    try:
        anchor_aff = build_anchor_vector(model, anchor_affordable_words)
        anchor_lq  = build_anchor_vector(model, anchor_low_quality_words)
    except ValueError as e:
        print(f"  Anchor building failed: {e}")
        return result

    sim_aff = cosine_between_vectors(context_vec, anchor_aff)
    sim_lq  = cosine_between_vectors(context_vec, anchor_lq)

    result["context_vector"]  = context_vec
    result["sim_affordable"]  = sim_aff
    result["sim_low_quality"] = sim_lq
    result["predicted_sense"] = "affordable" if sim_aff > sim_lq else "low-quality"

    return result


In [10]:

# Test sentences (cover both senses)
test_sentences = [
    "This phone is cheap and works perfectly for everyday use.",
    "The fabric feels cheap and tears apart after one wash.",
    "Amazingly cheap product for the features it offers.",
    "Looks cheap and the buttons broke within a week.",
    "Great cheap option for students on a tight budget.",
]

print("Q1(b): Context-based disambiguation of 'cheap'")
print()
print(f"Affordable  anchor words : {ANCHOR_AFFORDABLE}")
print(f"Low-quality anchor words : {ANCHOR_LOW_QUALITY}")
print()

for sentence in test_sentences:
    res = disambiguate_cheap(
        model_w2, sentence, PIVOT,
        ANCHOR_AFFORDABLE, ANCHOR_LOW_QUALITY
    )
    print(f"Sentence : {res['sentence']}")
    if res['sim_affordable'] is not None:
        print(f"  sim(context, affordable-anchor)  = {res['sim_affordable']:.4f}")
        print(f"  sim(context, low-quality-anchor) = {res['sim_low_quality']:.4f}")
        print(f"  Predicted sense : >>> {res['predicted_sense'].upper()} <<<")
    else:
        print("  Could not disambiguate (insufficient in-vocab context).")
    print()

Q1(b): Context-based disambiguation of 'cheap'

Affordable  anchor words : ['affordable', 'budget', 'value', 'inexpensive', 'economical']
Low-quality anchor words : ['flimsy', 'poor', 'inferior', 'shoddy', 'substandard']

  Anchor word 'affordable' not in vocab -- skipping.
  Anchor word 'budget' not in vocab -- skipping.
  Anchor word 'inexpensive' not in vocab -- skipping.
  Anchor word 'economical' not in vocab -- skipping.
  Anchor word 'flimsy' not in vocab -- skipping.
  Anchor word 'inferior' not in vocab -- skipping.
  Anchor word 'shoddy' not in vocab -- skipping.
  Anchor word 'substandard' not in vocab -- skipping.
Sentence : This phone is cheap and works perfectly for everyday use.
  sim(context, affordable-anchor)  = 0.5573
  sim(context, low-quality-anchor) = 0.3600
  Predicted sense : >>> AFFORDABLE <<<

  Anchor word 'affordable' not in vocab -- skipping.
  Anchor word 'budget' not in vocab -- skipping.
  Anchor word 'inexpensive' not in vocab -- skipping.
  Anchor word


### Mechanism:
  1. Remove the pivot word from the sentence.
  2. Average Word2Vec vectors of remaining context words.
  3. Compare the context centroid to two anchor centroids
     (affordable cluster vs low-quality cluster).
  4. The anchor with higher cosine similarity wins.

### Q1(c) -- Window size comparison: window=2 vs window=10

In [11]:
def compare_window_neighbours(model_small, model_large, probe_words, topn=8):
    """
    Print nearest neighbours for each probe word under both window settings.
    Useful for seeing whether the model captures syntactic vs semantic relations.
    """
    for word in probe_words:
        print(f"  Word: '{word}'")
        for label, model in [(f"window={WINDOW_SMALL} (syntactic)", model_small),
                             (f"window={WINDOW_LARGE} (semantic)",  model_large)]:
            if word not in model.wv.key_to_index:
                print(f"    [{label}] OOV")
                continue
            neighbours = model.wv.most_similar(word, topn=topn)
            words_only = ", ".join(w for w, _ in neighbours)
            print(f"    [{label}]\n      -> {words_only}")
        print()


def window_analogy_test(model_small, model_large, pos_words, neg_words):
    """
    Perform a word analogy query (e.g. king - man + woman) on both models.
    """
    for label, model in [(f"window={WINDOW_SMALL}", model_small),
                         (f"window={WINDOW_LARGE}", model_large)]:
        vocab = model.wv.key_to_index
        p_ok = [w for w in pos_words if w in vocab]
        n_ok = [w for w in neg_words if w in vocab]
        if not p_ok:
            print(f"  [{label}] positive words all OOV -- skipping analogy.")
            continue
        try:
            result = model.wv.most_similar(positive=p_ok, negative=n_ok, topn=5)
            words_only = ", ".join(w for w, _ in result)
            print(f"  [{label}] analogy result: {words_only}")
        except Exception as e:
            print(f"  [{label}] analogy failed: {e}")


# Train the large-window model
model_w10 = train_word2vec(
    tokenised,
    vector_size=W2V_VECTOR_SIZE,
    window=WINDOW_LARGE,
    min_count=W2V_MIN_COUNT,
    epochs=W2V_EPOCHS,
    seed=RANDOM_SEED,
)



Word2Vec trained | window=10 | vocab size=249


In [12]:
print()
print("Q1(c): window=2 vs window=10")


probe_words = ["battery", "quality", "cheap", "product", "good"]

print()
print("Nearest neighbours comparison:")
print("-" * 60)
compare_window_neighbours(model_w2, model_w10, probe_words, topn=8)

print("Analogy test -- 'good - quality + poor' (semantic relation):")
window_analogy_test(model_w2, model_w10,
                    pos_words=["good", "poor"],
                    neg_words=["quality"])

print()
print("Summary:")
print(f"  window={WINDOW_SMALL} (small): captures SYNTACTIC relationships.")
print("    Nearby words tend to be same POS, co-occurring modifiers, etc.")
print(f"  window={WINDOW_LARGE} (large): captures SEMANTIC/TOPICAL relationships.")
print("    Words from the same topic cluster appear even across longer spans.")
print("  For sentiment/NLP tasks, window=5-10 is usually preferred.")
print("  For syntactic tasks (POS, parsing), window=2-3 tends to be better.")


Q1(c): window=2 vs window=10

Nearest neighbours comparison:
------------------------------------------------------------
  Word: 'battery'
    [window=2 (syntactic)]
      -> backup, camera, bhi, theek, bohot, bas, amazing, thaak
    [window=10 (semantic)]
      -> backup, bhi, camera, bohot, theek, recommended, ye, thaak

  Word: 'quality'
    [window=2 (syntactic)]
      -> satisfied, thoda, excellent, highly, zyada, bohot, consistent, top
    [window=10 (semantic)]
      -> satisfied, price, again, is, sure, third, buy, poor

  Word: 'cheap'
    [window=2 (syntactic)]
      -> material, finishing, price, worth, thoda, poor, zyada, control
    [window=10 (semantic)]
      -> material, finishing, much, poor, better, price, worth, expected

  Word: 'product'
    [window=2 (syntactic)]
      -> extremely, zyada, control, hafte, cheap, are, mein, thoda
    [window=10 (semantic)]
      -> recommended, to, ek, too, paisa, overall, return, mein

  Word: 'good'
    [window=2 (syntactic)]
 

---
## Q2 -- Similarity of Review A and Review B Across Four Representations
### Q2 Setup

In [13]:
print("Review A:", REVIEW_A)
print("Review B:", REVIEW_B)
print()
print("Both express the same MIXED sentiment:")
print("  positive aspect  -> camera / photos")
print("  negative aspect  -> battery")
print("Ideal similarity score should be HIGH.")

Review A: incredible camera but terrible battery life
Review B: Battery drains fast, although photos are stunning

Both express the same MIXED sentiment:
  positive aspect  -> camera / photos
  negative aspect  -> battery
Ideal similarity score should be HIGH.


### Q2 Method 1 -- Bag-of-Words

In [14]:
def compute_bow_similarity(text_a, text_b):
    """
    Compute cosine similarity between two texts using Bag-of-Words vectors.
    """
    vectorizer = CountVectorizer()
    matrix = vectorizer.fit_transform([text_a, text_b]).toarray()
    vocab  = vectorizer.get_feature_names_out().tolist()

    vec_a = matrix[0]
    vec_b = matrix[1]

    sim = float(cosine_similarity(matrix[0:1], matrix[1:2])[0, 0])

    # Words that appear in both
    overlap = [vocab[i] for i in range(len(vocab))
               if vec_a[i] > 0 and vec_b[i] > 0]

    return {
        "similarity": sim,
        "vocab": vocab,
        "vector_a": vec_a,
        "vector_b": vec_b,
        "overlap_words": overlap,
    }


bow_result = compute_bow_similarity(REVIEW_A, REVIEW_B)

print("Method 1: Bag-of-Words (BOW)")
print(f"  Vocabulary    : {bow_result['vocab']}")
print(f"  Vector A      : {bow_result['vector_a']}")
print(f"  Vector B      : {bow_result['vector_b']}")
print(f"  Overlapping words : {bow_result['overlap_words']}")
print(f"  Cosine similarity : {bow_result['similarity']:.4f}")
print()
print("Q2(b) -- Exact word overlap analysis:")
tokens_a = set(REVIEW_A.lower().split())
tokens_b = set(REVIEW_B.lower().split())
shared   = tokens_a & tokens_b
only_a   = tokens_a - tokens_b
only_b   = tokens_b - tokens_a
print(f"  Tokens in A only : {sorted(only_a)}")
print(f"  Tokens in B only : {sorted(only_b)}")
print(f"  Shared tokens    : {sorted(shared)}")
print()
print("  'incredible' and 'stunning' both mean impressive -- but are different words.")
print("  'terrible' and 'drains fast' both signal bad battery -- again different tokens.")
print("  BOW sees almost NO overlap -> low similarity despite matching sentiment.")
print("  This is the BOW failure: it is purely lexical, ignoring meaning.")

Method 1: Bag-of-Words (BOW)
  Vocabulary    : ['although', 'are', 'battery', 'but', 'camera', 'drains', 'fast', 'incredible', 'life', 'photos', 'stunning', 'terrible']
  Vector A      : [0 0 1 1 1 0 0 1 1 0 0 1]
  Vector B      : [1 1 1 0 0 1 1 0 0 1 1 0]
  Overlapping words : ['battery']
  Cosine similarity : 0.1543

Q2(b) -- Exact word overlap analysis:
  Tokens in A only : ['but', 'camera', 'incredible', 'life', 'terrible']
  Tokens in B only : ['although', 'are', 'drains', 'fast,', 'photos', 'stunning']
  Shared tokens    : ['battery']

  'incredible' and 'stunning' both mean impressive -- but are different words.
  'terrible' and 'drains fast' both signal bad battery -- again different tokens.
  BOW sees almost NO overlap -> low similarity despite matching sentiment.
  This is the BOW failure: it is purely lexical, ignoring meaning.


### Q2 Method 2 -- TF-IDF

In [15]:
def compute_tfidf_similarity(text_a, text_b, corpus_texts=None):
    """
    Compute cosine similarity between two texts using TF-IDF vectors.
    If corpus_texts is provided, the IDF is fitted on the full corpus;
    otherwise it is fitted only on the two input texts.

    """
    fit_corpus = corpus_texts if corpus_texts else [text_a, text_b]
    vectorizer = TfidfVectorizer()
    vectorizer.fit(fit_corpus)

    matrix = vectorizer.transform([text_a, text_b])
    sim    = float(cosine_similarity(matrix[0:1], matrix[1:2])[0, 0])

    feature_names = vectorizer.get_feature_names_out()

    def top_terms(vec, n=5):
        arr = vec.toarray().flatten()
        indices = np.argsort(arr)[::-1][:n]
        return [(feature_names[i], round(float(arr[i]), 4)) for i in indices if arr[i] > 0]

    return {
        "similarity": sim,
        "top_terms_a": top_terms(matrix[0]),
        "top_terms_b": top_terms(matrix[1]),
    }


# Fit on the full corpus for realistic IDF
corpus_for_idf = reviews_df["review_text"].tolist()
tfidf_result   = compute_tfidf_similarity(REVIEW_A, REVIEW_B, corpus_texts=corpus_for_idf)

print("Method 2: TF-IDF")
print(f"  Top TF-IDF terms in A : {tfidf_result['top_terms_a']}")
print(f"  Top TF-IDF terms in B : {tfidf_result['top_terms_b']}")
print(f"  Cosine similarity     : {tfidf_result['similarity']:.4f}")
print()
print("  TF-IDF downweights common words (the, but, although) and upweights")
print("  rare words. However it still requires IDENTICAL tokens to match.")
print("  'incredible' and 'stunning' remain separate dimensions -> low overlap.")

Method 2: TF-IDF
  Top TF-IDF terms in A : [('battery', 0.5957), ('camera', 0.5957), ('terrible', 0.4326), ('but', 0.3212)]
  Top TF-IDF terms in B : [('battery', 0.714), ('are', 0.5965), ('fast', 0.3666)]
  Cosine similarity     : 0.4253

  TF-IDF downweights common words (the, but, although) and upweights
  rare words. However it still requires IDENTICAL tokens to match.
  'incredible' and 'stunning' remain separate dimensions -> low overlap.


### Q2 Method 3 -- Word2Vec Averaging

In [16]:
def compute_w2v_average_similarity(model, text_a, text_b):
    """
    Represent each text as the mean of its token Word2Vec vectors, then
    compute cosine similarity between the two means.
    """
    def average_vector(text):
        tokens = simple_preprocess(clean_text(text), deacc=True)
        vecs   = []
        oov    = []
        for t in tokens:
            if t in model.wv.key_to_index:
                vecs.append(model.wv[t])
            else:
                oov.append(t)
        if not vecs:
            return None, oov
        return np.mean(vecs, axis=0), oov

    vec_a, oov_a = average_vector(text_a)
    vec_b, oov_b = average_vector(text_b)

    if vec_a is None or vec_b is None:
        return {"similarity": None, "oov_a": oov_a, "oov_b": oov_b}

    sim = cosine_between_vectors(vec_a, vec_b)
    return {"similarity": sim, "oov_a": oov_a, "oov_b": oov_b}


w2v_result = compute_w2v_average_similarity(model_w2, REVIEW_A, REVIEW_B)


print("Method 3: Word2Vec Averaging")

print(f"  OOV tokens in A : {w2v_result['oov_a']}")
print(f"  OOV tokens in B : {w2v_result['oov_b']}")
if w2v_result['similarity'] is not None:
    print(f"  Cosine similarity : {w2v_result['similarity']:.4f}")
else:
    print("  Could not compute -- all tokens OOV.")
print()
print("  Word2Vec closes the semantic gap partially.")
print("  'incredible' and 'stunning' will be nearby in embedding space.")
print("  Limitation: averaging loses word order and negation structure.")

Method 3: Word2Vec Averaging
  OOV tokens in A : ['incredible', 'life']
  OOV tokens in B : ['drains', 'although', 'photos', 'stunning']
  Cosine similarity : 0.6208

  Word2Vec closes the semantic gap partially.
  'incredible' and 'stunning' will be nearby in embedding space.
  Limitation: averaging loses word order and negation structure.


### Q2 Method 4 - Sentence-BERT

In [17]:
def compute_sbert_similarity(model_name, text_a, text_b):
    """
    Encode two texts with a Sentence-BERT model and return their cosine similarity.
    """
    try:
        sbert_model = SentenceTransformer(model_name)
    except Exception as e:
        print(f"  Could not load SBERT model '{model_name}': {e}")
        return {"similarity": None, "embedding_dim": None}

    embeddings = sbert_model.encode([text_a, text_b], convert_to_numpy=True)
    sim = float(cosine_similarity(embeddings[0:1], embeddings[1:2])[0, 0])

    return {
        "similarity": sim,
        "embedding_dim": embeddings.shape[1],
    }


sbert_result = compute_sbert_similarity(SBERT_MODEL_NAME, REVIEW_A, REVIEW_B)


print("Method 4: Sentence-BERT")

if sbert_result['similarity'] is not None:
    print(f"  Model           : {SBERT_MODEL_NAME}")
    print(f"  Embedding dim   : {sbert_result['embedding_dim']}")
    print(f"  Cosine similarity : {sbert_result['similarity']:.4f}")
else:
    print("  SBERT similarity could not be computed.")
print()
print("  SBERT encodes the ENTIRE sentence context using a transformer.")
print("  It understands that 'terrible battery' and 'drains fast' are")
print("  semantically equivalent, and so are 'incredible camera' and")
print("  'photos are stunning'. This gives the highest similarity.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8352.93it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Method 4: Sentence-BERT
  Model           : all-MiniLM-L6-v2
  Embedding dim   : 384
  Cosine similarity : 0.6336

  SBERT encodes the ENTIRE sentence context using a transformer.
  It understands that 'terrible battery' and 'drains fast' are
  semantically equivalent, and so are 'incredible camera' and
  'photos are stunning'. This gives the highest similarity.


### Q2 Summary -- All Four Methods + Semantic Gap Analysis

In [18]:
def print_similarity_summary(bow_res, tfidf_res, w2v_res, sbert_res):
    """
    Print a formatted comparison table of all four similarity methods.
    """
    rows = [
        ("BOW",          bow_res.get("similarity"),    "Lexical match only -- word counts"),
        ("TF-IDF",       tfidf_res.get("similarity"),  "Weighted lexical -- rarer words matter more"),
        ("Word2Vec avg", w2v_res.get("similarity"),    "Static embeddings -- partial semantic bridge"),
        ("Sentence-BERT",sbert_res.get("similarity"),  "Contextual -- full sentence semantics"),
    ]


    print("Q2(a): Which representation correctly identifies similarity?")

    header = f"{'Method':<16}  {'Cosine Sim':>10}  Description"
    print(header)
    for method, sim, desc in rows:
        sim_str = f"{sim:.4f}" if sim is not None else "  N/A  "
        print(f"{method:<16}  {sim_str:>10}  {desc}")

    print()
    print("Q2(c): The 'Semantic Gap' -- how each method closes it:")
    print("  BOW          : No gap closure. Each word is an independent dimension.")
    print("                 'incredible' and 'stunning' are orthogonal -> no match.")
    print()
    print("  TF-IDF       : Marginal improvement via IDF weighting, but still")
    print("                 purely token-surface. The gap remains wide.")
    print()
    print("  Word2Vec avg : Significant gap closure. 'incredible' and 'stunning'")
    print("                 are close in embedding space (trained on large text).")
    print("                 Residual gap: averaging destroys word-order information.")
    print()
    print("  Sentence-BERT: Nearly full closure. Transformer attention lets the model")
    print("                 see that 'terrible battery life' and 'battery drains fast'")
    print("                 carry the same meaning, and so do the positive phrases.")
    print("                 This is the best representation for semantic similarity.")


print_similarity_summary(bow_result, tfidf_result, w2v_result, sbert_result)

Q2(a): Which representation correctly identifies similarity?
Method            Cosine Sim  Description
BOW                   0.1543  Lexical match only -- word counts
TF-IDF                0.4253  Weighted lexical -- rarer words matter more
Word2Vec avg          0.6208  Static embeddings -- partial semantic bridge
Sentence-BERT         0.6336  Contextual -- full sentence semantics

Q2(c): The 'Semantic Gap' -- how each method closes it:
  BOW          : No gap closure. Each word is an independent dimension.
                 'incredible' and 'stunning' are orthogonal -> no match.

  TF-IDF       : Marginal improvement via IDF weighting, but still
                 purely token-surface. The gap remains wide.

  Word2Vec avg : Significant gap closure. 'incredible' and 'stunning'
                 are close in embedding space (trained on large text).
                 Residual gap: averaging destroys word-order information.

  Sentence-BERT: Nearly full closure. Transformer attention lets the